# Prodigy_Audit_R27 — Observability Audit retro-attivo R25 + R26

**Obiettivo**: audit retro-attivo dei 24 checkpoint R25 (18) + R26 (6) con metriche estese R27:
1. **`val_T_intra_corr`** — Pearson(T_pred, T_true) DOPO rimozione media per-sample. Disambigua se T_corr=0.5 di B1 è cross-driver (corr. tra T_avg per scenario diverso) o vero intra-driver (cattura dinamica T(t) entro scenario).
2. **`rank_effective`** + **`cond_number`** su `Cov(decoded_params)` (5×5) raccolti su val set. Misura la dimensionalità effettiva usata dal decoder. Atteso da analisi: rank ~3 invece di 5 (rank-collapse).
3. **Fix bug LAYER_MAP** — 4/6 colonne gradient erano NaN su tutti i CSV pregress. Da R27 in poi `gn_hidden_fc`, `gn_hidden_base_threshold`, `gn_hidden_thresh_jump`, `gn_out_fc` saranno popolate.

**Decisioni che escono da R27**:
- Quale run è il champion **reale** con le nuove metriche? (B1 resta vincente? F4 sale?)
- Il decoder ha rank ~3 (rank-collapse confermato) o 5 (l'identificabilità è vera causa)?
- T_intra_corr discrimina puramente cross-driver da intra-driver?

**Nota**: nessun training nuovo. Solo re-valuta + aggrega.

**Output**: `results/Prodigy_Study/Audit_R27/audit_summary.csv` + `audit_metrics.json` in ogni run dir.

In [ ]:
# Cell 1 -- Bootstrap + ENV check (R27)
import sys, os, subprocess
import importlib.util as _imu

for pkg in ['prodigyopt', 'pandas', 'matplotlib', 'seaborn']:
    if _imu.find_spec(pkg) is None:
        print(f'  installing {pkg}...')
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg], check=True)

for f in ['train.py', 'core/network.py', 'scripts/audit_checkpoints.py',
          'results/Prodigy_Study/Ablation_R25/_aggregate_full.csv']:
    assert os.path.isfile(f), f'MISSING: {f}'
    print(f'  [OK] {f}')

sys.path.insert(0, '.')
for mod in ['train','core.network']:
    if mod in sys.modules: del sys.modules[mod]

# Verifica R27 changes attive
from train import CSVLogger, BatchCSVLogger, _make_batch_row, val_epoch
assert 'val_T_intra_corr' in CSVLogger.COLS, 'R27 BUG: val_T_intra_corr non in CSVLogger.COLS'
print('  [OK] val_T_intra_corr in CSVLogger.COLS')

# Verifica LAYER_MAP fix: smoke su modello baseline -> 6/6 cols popolati
import torch
from core.network import CF_FSNN_Net
torch.manual_seed(42)
m = CF_FSNN_Net(hidden_size=32, rank=8, max_delay=6, bit_shift=3)
x = torch.rand(2, 10, 4)
ps, sr = m.forward_sequence_with_stats(x)
(ps.mean() + sr.mean()).backward()
pre_norms = {n: p.grad.detach().norm().item() for n,p in m.named_parameters() if p.grad is not None}
class _Opt: param_groups=[{'lr':1.0}]
comps = {'total':0.5,'data':0.4,'phys':0.1,'ou':0.01,'bc':0.0,'sr':0.05,'T_aux':0.0}
row = _make_batch_row(1,1,comps,0.1,pre_norms,1.0,1.0,m,_Opt(),grad_extras={})
_six = ['gn_hidden_fc','gn_hidden_recU','gn_hidden_recV','gn_hidden_base_threshold','gn_hidden_thresh_jump','gn_out_fc']
import math
nan_cols = [c for c in _six if not math.isfinite(row.get(c, float('nan')))]
assert not nan_cols, f'R27 BUG: colonne ancora NaN: {nan_cols}'
print(f'  [OK] LAYER_MAP fix: 6/6 colonne gradient popolate')

# audit_checkpoints.py importabile
from scripts.audit_checkpoints import discover_runs, find_checkpoint
print('  [OK] audit_checkpoints.py importabile')

br = subprocess.run(['git','branch','--show-current'], capture_output=True, text=True).stdout.strip()
print(f'\n[GIT] branch: {br}')
assert br == 'Prodigy_Deep_Study'
print('\nENV check passed.')

In [ ]:
# Cell 2 -- Discover + run audit_checkpoints.py su R25+R26 (filter ^R2[5-6]_)
import subprocess, sys
from pathlib import Path
from scripts.audit_checkpoints import discover_runs, find_checkpoint

results_root = Path('results/Prodigy_Study')
all_runs = discover_runs(results_root)
import re
rx = re.compile(r'^R2[5-6]_')
r25r26 = [r for r in all_runs if rx.search(r.name)]
print(f'Totale run R25+R26: {len(r25r26)} (atteso ~24)')

# Quanti hanno checkpoint disponibile?
available = [(r, find_checkpoint(r.name, results_root)) for r in r25r26]
with_ckpt = [(r,c) for (r,c) in available if c is not None]
without = [r.name for (r,c) in available if c is None]
print(f'  con checkpoint: {len(with_ckpt)}')
print(f'  senza checkpoint: {len(without)}')
if without:
    print('  SKIP:', without[:10], '...' if len(without)>10 else '')

# Esegui audit (subprocess per isolamento + log pulito)
OUTPUT_CSV = 'results/Prodigy_Study/Audit_R27/audit_summary.csv'
cmd = [sys.executable, 'scripts/audit_checkpoints.py',
       '--results_root', 'results/Prodigy_Study',
       '--output_csv', OUTPUT_CSV,
       '--pattern', r'^R2[5-6]_',
       '--device', 'cpu']
print(f'\nRunning: {" ".join(cmd)}')
r = subprocess.run(cmd, capture_output=True, text=True, encoding='utf-8', errors='replace')
print(r.stdout)
if r.returncode != 0:
    print('STDERR:', r.stderr[-2000:])
assert r.returncode == 0, 'audit_checkpoints.py exit non-zero'

In [ ]:
# Cell 3 -- Aggregator + tabella comparativa (focus: cross vs intra T_corr)
import pandas as pd, numpy as np
from IPython.display import display, Markdown

df = pd.read_csv(OUTPUT_CSV)
print(f'Run auditati: {len(df)}')

# Sintesi top metrics
key_cols = ['tag','val_val_data','val_val_T_tracking_corr','val_val_T_intra_corr',
            'rank_effective','cond_number','val_val_v0_pred_mean','val_val_s0_pred_mean',
            'cfg_lambda_T_aux','cfg_cf_max_delay','cfg_lambda_sr']
show_cols = [c for c in key_cols if c in df.columns]
display(Markdown('## Tabella completa (best epoch, ordine: val_T_intra_corr desc)'))
display(df.sort_values('val_val_T_intra_corr', ascending=False)[show_cols].round(4))

# Confronto B1 vs altri (R25_B1 era il champion R25 dichiarato)
display(Markdown('## Confronto R25_B1 (vecchio champion) vs altri'))
b1 = df[df.tag.str.contains('B1_T_aux_low', na=False)]
if not b1.empty:
    b1_row = b1.iloc[0]
    print(f"R25_B1 metrics:")
    print(f"  val_data        = {b1_row['val_val_data']:.4f}")
    print(f"  T_tracking_corr = {b1_row['val_val_T_tracking_corr']:.3f}")
    print(f"  T_intra_corr    = {b1_row['val_val_T_intra_corr']:.3f}  <-- NEW R27")
    print(f"  rank_effective  = {b1_row['rank_effective']}/5")
    print(f"  cond_number     = {b1_row['cond_number']:.2e}")

    # Verdetto: era cross-driver illusion o vera dinamica?
    ratio = b1_row['val_val_T_intra_corr'] / max(b1_row['val_val_T_tracking_corr'], 1e-6)
    if ratio > 0.7:
        display(Markdown(f'**Verdetto B1**: T_intra/T_tracking = {ratio:.2f} → SEGNALE GENUINO intra-driver, B1 è davvero champion.'))
    elif ratio > 0.3:
        display(Markdown(f'**Verdetto B1**: T_intra/T_tracking = {ratio:.2f} → PARZIALE. T_corr era misto cross+intra.'))
    else:
        display(Markdown(f'**Verdetto B1**: T_intra/T_tracking = {ratio:.2f} → ILLUSIONE cross-driver, B1 NON era vero champion intra. Riapertura.'))

# Top per metrica diversa
for metric, label in [('val_val_T_intra_corr', 'T_intra_corr'),
                       ('val_val_T_tracking_corr', 'T_tracking_corr'),
                       ('val_val_data', 'val_data (min)')]:
    if metric in df.columns:
        asc = ('min' in label)
        top = df.sort_values(metric, ascending=asc).head(3)
        display(Markdown(f'## Top 3 per {label}'))
        display(top[['tag', metric, 'rank_effective', 'cfg_lambda_T_aux', 'cfg_cf_max_delay']].round(4))

In [ ]:
# Cell 4 -- Plot: rank-collapse + scatter T_corr cross vs intra
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Plot 1: scatter T_tracking_corr vs T_intra_corr
ax = axes[0]
ax.scatter(df['val_val_T_tracking_corr'], df['val_val_T_intra_corr'],
           s=80, alpha=0.7, edgecolor='black')
ax.plot([0, 1], [0, 1], 'k--', alpha=0.3, label='y=x (identity)')
ax.axhline(0.3, color='red', linestyle=':', alpha=0.5, label='intra=0.3 (soglia weak)')
for _, r in df.iterrows():
    short = r['tag'].replace('R25_','').replace('R26_','').replace('_replica','')[:12]
    ax.annotate(short, (r['val_val_T_tracking_corr'], r['val_val_T_intra_corr']),
                fontsize=7, alpha=0.7)
ax.set_xlabel('val_T_tracking_corr (cross+intra mixed)')
ax.set_ylabel('val_T_intra_corr (intra only)')
ax.set_title('R27 Disambiguation: cross-driver illusion vs vera dinamica')
ax.legend(); ax.grid(alpha=0.3)

# Plot 2: distribuzione rank_effective
ax = axes[1]
ax.hist(df['rank_effective'].dropna(), bins=np.arange(0.5, 6.5), edgecolor='black', alpha=0.7)
ax.axvline(5, color='green', linestyle='--', label='rank=5 (full)')
ax.axvline(3, color='orange', linestyle='--', label='rank=3 (collapse atteso)')
ax.set_xlabel('rank_effective Cov(decoded_params)')
ax.set_ylabel('# run')
ax.set_title('R27: Rank-collapse del decoder')
ax.legend(); ax.grid(alpha=0.3)

# Plot 3: spectrum singular values normalizzati (mean across runs)
ax = axes[2]
sv_cols = [f'sv_norm_{i+1}' for i in range(5) if f'sv_norm_{i+1}' in df.columns]
if sv_cols:
    sv_mean = df[sv_cols].mean().values
    sv_std  = df[sv_cols].std().values
    ax.errorbar(range(1, len(sv_cols)+1), sv_mean, yerr=sv_std,
                marker='o', capsize=5, linewidth=2)
    ax.axhline(0.01, color='red', linestyle=':', alpha=0.5, label='soglia rank (1%)')
    ax.set_yscale('log')
    ax.set_xlabel('Singular value index')
    ax.set_ylabel('SV normalized (mean ± std across runs)')
    ax.set_title('R27: Spettro singular values Cov decoder')
    ax.legend(); ax.grid(alpha=0.3, which='both')

plt.tight_layout()
plt.savefig('results/Prodigy_Study/Audit_R27/R27_diagnostic_plots.png', dpi=110)
plt.show()
print('Salvato: results/Prodigy_Study/Audit_R27/R27_diagnostic_plots.png')

In [ ]:
# Cell 5 -- Push git (audit_summary.csv + diagnostic_plots + audit_metrics.json per run)
import subprocess, os, tempfile
RESULTS_DIR = 'results/Prodigy_Study/Audit_R27'

# Stage paths: nuove cartella Audit_R27 + audit_metrics.json sparsi nei run R25/R26
to_add = [RESULTS_DIR]
import glob
for jp in glob.glob('results/Prodigy_Study/**/audit_metrics.json', recursive=True):
    to_add.append(jp)

subprocess.run(['git','add'] + to_add, check=True)

# Stats per commit message
import pandas as pd
df = pd.read_csv(f'{RESULTS_DIR}/audit_summary.csv')
n = len(df)
best_intra = df.sort_values('val_val_T_intra_corr', ascending=False).iloc[0]
best_data  = df.sort_values('val_val_data', ascending=True).iloc[0]
mean_rank  = df['rank_effective'].mean()

msg = f"""results (R27 Audit): {n} run R25+R26 auditati con metriche estese

Metriche aggiunte:
- val_T_intra_corr (Pearson mean-removed per-sample)
- rank_effective Cov(decoded_params)
- cond_number + singular_values normalizzati

Fix bug LAYER_MAP (train.py:704-722): 4 colonne gradient ora popolate
su tutti i futuri run (gn_hidden_fc, gn_hidden_base_threshold,
gn_hidden_thresh_jump, gn_out_fc).

Best T_intra_corr: {best_intra['tag']} = {best_intra['val_val_T_intra_corr']:.3f}
Best val_data:    {best_data['tag']} = {best_data['val_val_data']:.4f}
Rank effettivo medio: {mean_rank:.2f}/5
"""

with tempfile.NamedTemporaryFile('w', delete=False, suffix='.txt', encoding='utf-8') as fp:
    fp.write(msg)
    msg_path = fp.name

subprocess.run(['git','commit','-F', msg_path], check=True)
subprocess.run(['git','pull','--no-rebase','--no-edit','origin','Prodigy_Deep_Study'], check=True)
subprocess.run(['git','push','origin','Prodigy_Deep_Study'], check=True)
os.unlink(msg_path)
print('\n[OK] R27 Audit pushato su Prodigy_Deep_Study.')